# 03 — Hybrid: char TF-IDF + 37 handcrafted lexical features + LightGBM

**Improvement over the baseline:**
1. Combine sparse char-ngram TF-IDF with a dense vector of 37 handcrafted lexical/structural features (defined in `src/features.py`).
2. Train a **LightGBM** classifier (gradient-boosted trees) instead of Logistic Regression — captures non-linear interactions between TF-IDF tokens and handcrafted features.

**Output:** trained bundle saved to `models/hybrid_lgbm.joblib`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report,
                             roc_auc_score, average_precision_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

from features import extract_lexical_features, FEATURE_NAMES, normalize_url

RANDOM_STATE = 42
PROCESSED = Path('../data/processed/phiusiil_clean.parquet')
MODELS = Path('../models'); MODELS.mkdir(exist_ok=True)
print('Handcrafted feature count:', len(FEATURE_NAMES))

Handcrafted feature count: 37


## Load, normalize to domain-level, augment, and split

Two data fixes before training:

1. **Normalize to `scheme://host`.** Raw PhiUSIIL leaks the label through URL
   structure (every legit URL is a bare `https://www.<domain>` crawl entry;
   phishing URLs often carry a path / lack `www`). A full-URL model exploits
   this and scores ~100% phishing for any normal URL with a path.
2. **Augment the legit class** with the top 30k Majestic Million domains. The
   PhiUSIIL legit set is a narrow crawl that excludes most mainstream sites
   (e.g. `github.com` appears *only* as phishing), so without this the model
   flags popular domains. Augmentation also breaks the residual `www` artifact.

We then drop ambiguous/duplicate hosts so no host appears in both train and test.

In [2]:
df = pd.read_parquet(PROCESSED)

# --- Domain-level normalization -------------------------------------------
# Raw PhiUSIIL leaks the label through URL structure: every legitimate URL is
# a bare https://www.<domain> crawl entry, while phishing URLs vary (often a
# path, often no www). A full-URL model just learns those crawl artifacts and
# scores ~100% phishing for any normal URL with a path. Collapse to the host.
df['URL'] = df['URL'].astype(str).map(normalize_url)

# --- Augment the legit class with popular real domains --------------------
# The legit class above is a homogeneous crawl, so the model never sees what
# mainstream domains look like (e.g. github.com appears in PhiUSIIL ONLY as
# phishing; wikipedia.org not at all). We add the top registered domains from
# the Majestic Million as legitimate. PhiUSIIL legit URLs are ALSO all https,
# which otherwise teaches "http -> phishing" (so even http://google.com scores
# 100% phishing); we therefore add each domain under BOTH http and https to
# decorrelate the scheme from the label. Top-list domains are authoritative-
# legit at the registered-domain level (they override conflicting PhiUSIIL rows).
AUG = Path('../data/external/top_legit_domains.txt')
top = [d.strip().lower() for d in AUG.read_text(encoding='utf-8').splitlines()
       if d.strip() and not d.startswith('#')]
aug_urls = [f'https://{d}' for d in top] + [f'http://{d}' for d in top]
aug = pd.DataFrame({'URL': aug_urls, 'label': 1})
df = df[~df['URL'].isin(set(aug['URL']))]
df = pd.concat([df[['URL', 'label']], aug], ignore_index=True)

# --- Domain-level dedup: drop ambiguous hosts, then exact duplicates -------
nun = df.groupby('URL')['label'].transform('nunique')
df = df[nun == 1].drop_duplicates(subset='URL').reset_index(drop=True)
print(f'Unique hosts after normalization + augmentation + dedup: {len(df):,}')
print(df['label'].value_counts())

X_urls = df['URL'].to_numpy()
y = df['label'].to_numpy()

X_tr_u, X_tmp_u, y_tr, y_tmp = train_test_split(
    X_urls, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_val_u, X_te_u, y_val, y_te = train_test_split(
    X_tmp_u, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE)
print(f'train={len(X_tr_u):,}  val={len(X_val_u):,}  test={len(X_te_u):,}')

Unique hosts after normalization + augmentation + dedup: 282,039
label
1    194831
0     87208
Name: count, dtype: int64
train=197,427  val=42,306  test=42,306


## Extract handcrafted features

In [3]:
def featurize_batch(urls):
    rows = [extract_lexical_features(u) for u in urls]
    return pd.DataFrame(rows, columns=FEATURE_NAMES).astype(np.float32)

%time F_tr  = featurize_batch(X_tr_u)
%time F_val = featurize_batch(X_val_u)
%time F_te  = featurize_batch(X_te_u)
F_tr.head()

CPU times: total: 9.67 s
Wall time: 9.8 s


CPU times: total: 2 s
Wall time: 2.02 s


CPU times: total: 1.91 s
Wall time: 1.92 s


,url_length,host_length,path_length,query_length,domain_length,subdomain_length,tld_length,num_dots,num_slashes,num_dashes,...,is_https,has_port,has_ip_host,has_at_symbol,has_double_slash_in_path,is_shortener,has_suspicious_keyword,num_suspicious_keywords,url_entropy,host_entropy
0,15.0,7.0,0.0,0.0,4.0,0.0,2.0,1.0,2.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.456565,2.807355
1,25.0,17.0,0.0,0.0,9.0,3.0,3.0,2.0,2.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.589275,2.771902
2,44.0,36.0,0.0,0.0,10.0,21.0,3.0,2.0,2.0,2.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.570354,4.384796
3,19.0,11.0,0.0,0.0,4.0,3.0,2.0,2.0,2.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.576618,2.663533
4,23.0,15.0,0.0,0.0,11.0,0.0,3.0,1.0,2.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.882045,3.506891


## Fit char-TF-IDF on training URLs

In [4]:
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=5,
    max_features=200_000,
    sublinear_tf=True,
)
%time T_tr  = vectorizer.fit_transform(X_tr_u)
T_val = vectorizer.transform(X_val_u)
T_te  = vectorizer.transform(X_te_u)
print('TF-IDF shape:', T_tr.shape)

CPU times: total: 15.7 s
Wall time: 15.8 s


TF-IDF shape: (197427, 187546)


## Standardise handcrafted features, then hstack with sparse TF-IDF

In [5]:
scaler = StandardScaler()
F_tr_s  = scaler.fit_transform(F_tr)
F_val_s = scaler.transform(F_val)
F_te_s  = scaler.transform(F_te)

X_tr  = hstack([T_tr,  csr_matrix(F_tr_s)]).tocsr()
X_val = hstack([T_val, csr_matrix(F_val_s)]).tocsr()
X_te  = hstack([T_te,  csr_matrix(F_te_s)]).tocsr()
print('Hybrid matrix shape:', X_tr.shape)

Hybrid matrix shape: (197427, 187583)


## Train LightGBM

In [6]:
clf = lgb.LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=20,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
%time clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)])

[LightGBM] [Info] Number of positive: 136381, number of negative: 61046


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 21.405278 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1921196


[LightGBM] [Info] Number of data points in the train set: 197427, number of used features: 53916


[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.690792 -> initscore=0.803825
[LightGBM] [Info] Start training from score 0.803825


Training until validation scores don't improve for 30 rounds


...

[500]	valid_0's binary_logloss: 0.0695633


[550]	valid_0's binary_logloss: 0.0693943


[600]	valid_0's binary_logloss: 0.0692993
Did not meet early stopping. Best iteration is:
[592]	valid_0's binary_logloss: 0.0692768


CPU times: total: 3h 20min 40s
Wall time: 15min 47s


,boosting_type,'gbdt'
,num_leaves,127
,max_depth,-1
,learning_rate,0.05
,n_estimators,600
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


## Evaluate

In [7]:
def report(name, y_true, y_pred, y_score):
    print(f'== {name} ==')
    print(f'Accuracy : {accuracy_score(y_true, y_pred):.4f}')
    print(f'ROC-AUC  : {roc_auc_score(y_true, y_score):.4f}')
    print(f'PR-AUC   : {average_precision_score(y_true, y_score):.4f}')
    print(classification_report(y_true, y_pred,
          target_names=['phish', 'legit'], digits=4))

y_val_score = clf.predict_proba(X_val)[:, 1]
y_val_pred  = (y_val_score >= 0.5).astype(int)
report('VAL',  y_val, y_val_pred, y_val_score)

y_te_score = clf.predict_proba(X_te)[:, 1]
y_te_pred  = (y_te_score >= 0.5).astype(int)
report('TEST', y_te, y_te_pred, y_te_score)

C:\Users\clark\OneDrive\Documents\GitHub\cybersecurity-finals\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


== VAL ==
Accuracy : 0.9789
ROC-AUC  : 0.9927
PR-AUC   : 0.9954
              precision    recall  f1-score   support

       phish     0.9899    0.9414    0.9650     13081
       legit     0.9743    0.9957    0.9849     29225

    accuracy                         0.9789     42306
   macro avg     0.9821    0.9685    0.9749     42306
weighted avg     0.9791    0.9789    0.9787     42306



C:\Users\clark\OneDrive\Documents\GitHub\cybersecurity-finals\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


== TEST ==
Accuracy : 0.9786
ROC-AUC  : 0.9929
PR-AUC   : 0.9957
              precision    recall  f1-score   support

       phish     0.9888    0.9415    0.9646     13081
       legit     0.9744    0.9952    0.9847     29225

    accuracy                         0.9786     42306
   macro avg     0.9816    0.9684    0.9746     42306
weighted avg     0.9788    0.9786    0.9785     42306



## Save model bundle

We save the vectorizer + scaler + classifier as a single dict so the Streamlit app can run end-to-end inference from a single file.

In [8]:
bundle = {
    'vectorizer': vectorizer,
    'scaler': scaler,
    'classifier': clf,
    'feature_names': FEATURE_NAMES,
}
out = MODELS / 'hybrid_lgbm.joblib'
joblib.dump(bundle, out)
print(f'Saved -> {out} ({out.stat().st_size / 1e6:.1f} MB)')

Saved -> ..\models\hybrid_lgbm.joblib (19.7 MB)
